## 2. Choosing Relevant Variables

### Economic 

In [ ]:
# 1. Define indicators to keep
economic_selected = economic_data[['Country Name', 'Country Code', 'Series Name','2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']].copy()

# Keep only selected variables
economic_selected = economic_selected[
    economic_selected['Series Name'].isin([
        'GDP per capita (current US$)',
        'Inflation, consumer prices (annual %)',
        'Unemployment, total (% of total labor force) (modeled ILO estimate)'
    ])
].copy()

# Convert values to numeric
year_columns = ['2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]', '2024 [YR2024]']
for col in year_columns:
    economic_selected[col] = pd.to_numeric(economic_selected[col], errors='coerce')

# Use 2024 first, if missing use 2023
economic_selected['selected_value'] = (economic_selected['2024 [YR2024]']
    .fillna(economic_selected['2023 [YR2023]'])
    .fillna(economic_selected['2022 [YR2022]'])
    .fillna(economic_selected['2021 [YR2021]'])
    .fillna(economic_selected['2020 [YR2020]'])
)

# Convert to wide format
economic_final = economic_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

# Rename columns
economic_final = economic_final.rename(columns={
    'Country Name': 'Country',
    'GDP per capita (current US$)': 'gdp_per_capita_recent',
    'Inflation, consumer prices (annual %)': 'inflation_recent',
    'Unemployment, total (% of total labor force) (modeled ILO estimate)': 'unemployment_recent'
})

economic_final.columns.name = None

print(economic_final.shape)
economic_final.head(211)

- 2024 was used as the reference year. Where 2024 values were missing, 2023 values were used as a one-year backfill.

## Health and Education

In [ ]:
health_education_selected = health_education_data[['Country Name', 'Country Code', 'Series Name','2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']].copy()

health_education_selected = health_education_selected[
    health_education_selected['Series Name'].isin([
        'Life expectancy at birth, total (years)',
        'School enrollment, secondary (% gross)',
    ])
].copy()

# Convert values to numeric
year_columns = ['2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']

for col in year_columns:
    health_education_selected[col] = pd.to_numeric(
        health_education_selected[col],
        errors='coerce'
    )

# Create selected value column
health_education_selected['selected_value'] = np.nan

# Life expectancy: use 2024 first, if missing use 2023
life_mask = health_education_selected['Series Name'] == 'Life expectancy at birth, total (years)'
health_education_selected.loc[life_mask, 'selected_value'] = (
    health_education_selected.loc[life_mask, '2024 [YR2024]']
    .fillna(health_education_selected.loc[life_mask, '2023 [YR2023]'])
)

# Secondary enrollment: latest available from 2024 to 2020
secondary_mask = health_education_selected['Series Name'] == 'School enrollment, secondary (% gross)'
health_education_selected.loc[secondary_mask, 'selected_value'] = (
    health_education_selected.loc[secondary_mask, '2024 [YR2024]']
    .fillna(health_education_selected.loc[secondary_mask, '2023 [YR2023]'])
    .fillna(health_education_selected.loc[secondary_mask, '2022 [YR2022]'])
    .fillna(health_education_selected.loc[secondary_mask, '2021 [YR2021]'])
    .fillna(health_education_selected.loc[secondary_mask, '2020 [YR2020]'])
)

# Pivot to wide format
health_education_final = health_education_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

health_education_final = health_education_final.rename(columns={
    'Country Name': 'Country',
    'Life expectancy at birth, total (years)': 'life_expectancy_2024',
    'School enrollment, secondary (% gross)': 'secondary_enrollment_recent',
})

health_education_final.columns.name = None

print(health_education_final.shape)
health_education_final.head(217)

## Infrastructure

In [ ]:
infrastructure_selected = infrastructure_data[['Country Name', 'Country Code', 'Series Name', '2023 [YR2023]', '2024 [YR2024]']].copy()

infrastructure_selected = infrastructure_selected[
    infrastructure_selected['Series Name'].isin([
        'Access to electricity (% of population)',
        'People using at least basic drinking water services (% of population)',
        'People using at least basic sanitation services (% of population)'
    ])
].copy()

infrastructure_selected['2023 [YR2023]'] = pd.to_numeric(infrastructure_selected['2023 [YR2023]'], errors='coerce')
infrastructure_selected['2024 [YR2024]'] = pd.to_numeric(infrastructure_selected['2024 [YR2024]'], errors='coerce')

infrastructure_selected['selected_value'] = infrastructure_selected['2024 [YR2024]'].fillna(infrastructure_selected['2023 [YR2023]'])

infrastructure_final = infrastructure_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

infrastructure_final = infrastructure_final.rename(columns={
    'Country Name': 'Country',
    'Access to electricity (% of population)': 'electricity_access_2024',
    'People using at least basic drinking water services (% of population)': 'drinking_water_access',
    'People using at least basic sanitation services (% of population)': 'sanitation_access'
})

infrastructure_final.columns.name = None

print(infrastructure_final.shape)
infrastructure_final.head(216)

## Environment

In [ ]:
environment_selected = environmental_data[['Country Name', 'Country Code', 'Series Name', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]']].copy()

environment_selected = environment_selected[
    environment_selected['Series Name'].isin([
        'Forest area (% of land area)',
        'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)',
        'Renewable energy consumption (% of total final energy consumption)'
    ])
].copy()

# Convert values to numeric
environment_selected['2020 [YR2020]'] = pd.to_numeric(environment_selected['2020 [YR2020]'], errors='coerce')
environment_selected['2021 [YR2021]'] = pd.to_numeric(environment_selected['2021 [YR2021]'], errors='coerce')
environment_selected['2022 [YR2022]'] = pd.to_numeric(environment_selected['2022 [YR2022]'], errors='coerce')
environment_selected['2023 [YR2023]'] = pd.to_numeric(environment_selected['2023 [YR2023]'], errors='coerce')

# Create selected value column
environment_selected['selected_value'] = np.nan

# Forest area: use 2023 first, if missing use 2022
forest_mask = environment_selected['Series Name'] == 'Forest area (% of land area)'
environment_selected.loc[forest_mask, 'selected_value'] = (
    environment_selected.loc[forest_mask, '2023 [YR2023]']
    .fillna(environment_selected.loc[forest_mask, '2022 [YR2022]'])
)

# PM2.5: use 2020 because only 2020 data is available for most countries.
pm25_mask = environment_selected['Series Name'] == 'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)'
environment_selected.loc[pm25_mask, 'selected_value'] = environment_selected.loc[pm25_mask, '2020 [YR2020]']

# Renewable energy: use 2022 first, if missing use 2021
renewable_mask = environment_selected['Series Name'] == 'Renewable energy consumption (% of total final energy consumption)'
environment_selected.loc[renewable_mask, 'selected_value'] = (
    environment_selected.loc[renewable_mask, '2022 [YR2022]']
    .fillna(environment_selected.loc[renewable_mask, '2021 [YR2021]'])
)

environment_final = environment_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

environment_final = environment_final.rename(columns={
    'Country Name': 'Country',
    'Forest area (% of land area)': 'forest_area_2023_reference',
    'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)': 'pm2.5_exposure_2020',
    'Renewable energy consumption (% of total final energy consumption)': 'renewable_energy_recent'
})

environment_final.columns.name = None

print(environment_final.shape)
environment_final.head(215)

## Governance

In [ ]:
governance_selected = governance_data[['Country Name', 'Country Code', 'Series Name', '2023 [YR2023]', '2024 [YR2024]']].copy()

governance_selected = governance_selected[
    governance_selected['Series Name'].isin([
        'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)',
        'Rule of Law - Governance estimate (approx. -2.5 to +2.5)',
        'Political Stability - Governance estimate (approx. -2.5 to +2.5)'
    ])
].copy()

# Convert values to numeric
governance_selected['2023 [YR2023]'] = pd.to_numeric(governance_selected['2023 [YR2023]'], errors='coerce')
governance_selected['2024 [YR2024]'] = pd.to_numeric(governance_selected['2024 [YR2024]'], errors='coerce')
governance_selected['selected_value'] = governance_selected['2024 [YR2024]'].fillna(governance_selected['2023 [YR2023]'])

governance_final = governance_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

governance_final = governance_final.rename(columns={
    'Country Name': 'Country',
    'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)': 'government_effectiveness_2024',
    'Rule of Law - Governance estimate (approx. -2.5 to +2.5)': 'rule_of_law_2024',
    'Political Stability - Governance estimate (approx. -2.5 to +2.5)': 'political_stability_2024'
})

governance_final.columns.name = None

print(governance_final.shape)
governance_final.head()

## Combine Economic / Environment / Governance / Health & Education / Infrastructure indicators

In [ ]:
# Combine datasets
final_dataset_SLQI = economic_final.merge(
    health_education_final,
    on=['Country', 'Country Code'],
    how='outer'
)

final_dataset_SLQI = final_dataset_SLQI.merge(
    infrastructure_final,
    on=['Country', 'Country Code'],
    how='outer'
)

final_dataset_SLQI = final_dataset_SLQI.merge(
    governance_final,
    on=['Country', 'Country Code'],
    how='outer'
)

final_dataset_SLQI = final_dataset_SLQI.merge(
    environment_final,
    on=['Country', 'Country Code'],
    how='outer'
)

print(final_dataset_SLQI.shape)
final_dataset_SLQI.head()

## Check missing values in final dataset

In [ ]:
print("Missing values in final dataset:")
print(final_dataset_SLQI.isnull().sum())

In [ ]:
# Count missing values for each country

indicator_columns = final_dataset_SLQI.columns.drop(['Country', 'Country Code'])

final_dataset_SLQI['missing_values_count'] = final_dataset_SLQI[indicator_columns].isnull().sum(axis=1)
final_dataset_SLQI['available_values_count'] = final_dataset_SLQI[indicator_columns].notnull().sum(axis=1)

final_dataset_SLQI[
    ['Country', 'Country Code', 'missing_values_count', 'available_values_count']
].head(20)

## Check missing percentage

In [ ]:
indicator_columns = final_dataset_SLQI.columns.drop(
    ['Country', 'Country Code', 'missing_values_count', 'available_values_count'],
    errors='ignore'
)

# Calculate missing percentage for each variable, will remove it if over 20%.
missing_summary = pd.DataFrame({
    'missing_count': final_dataset_SLQI[indicator_columns].isnull().sum(),
    'missing_percentage': final_dataset_SLQI[indicator_columns].isnull().sum() / len(final_dataset_SLQI) * 100
})

missing_summary = missing_summary.sort_values(by='missing_percentage', ascending=False)
missing_summary

- Tertiary enrolment was removed because it contained a high number of missing values even after using the latest available data from 2020 to 2024. 

In [ ]:
# Recalculate missing values for each country

indicator_columns = final_dataset_SLQI.columns.drop(
    ['Country', 'Country Code', 'missing_values_count', 'available_values_count'],
    errors='ignore'
)

final_dataset_SLQI['missing_values_count'] = final_dataset_SLQI[indicator_columns].isnull().sum(axis=1)
final_dataset_SLQI['available_values_count'] = final_dataset_SLQI[indicator_columns].notnull().sum(axis=1)

final_dataset_SLQI[
    ['Country', 'Country Code', 'missing_values_count', 'available_values_count']
].sort_values(by='missing_values_count', ascending=False).head(30)

In [ ]:
# Remove countries with too many missing values
max_missing_allowed = 4

final_dataset_clean = final_dataset_SLQI[
    final_dataset_SLQI['missing_values_count'] <= max_missing_allowed
].copy()

print(f"Original countries: {len(final_dataset_SLQI)}")
print(f"Countries after cleaning: {len(final_dataset_clean)}")

final_dataset_clean.head(199)

- Countries with more than 4 missing indicators were excluded. 

In [ ]:
print("Missing values after removing countries with too much missing data:")
print(final_dataset_clean.isnull().sum())

In [ ]:
# Fill remaining missing values with median

indicator_columns = final_dataset_clean.columns.drop(
    ['Country', 'Country Code', 'missing_values_count', 'available_values_count'],
    errors='ignore'
)

for col in indicator_columns:
    final_dataset_clean[col] = final_dataset_clean[col].fillna(
        final_dataset_clean[col].median()
    )

print("Missing values after median imputation:")
print(final_dataset_clean.isnull().sum())

- The remaining small number of missing values were filled using median imputation.

In [ ]:
final_dataset_clean = final_dataset_clean.drop(
    columns=['missing_values_count', 'available_values_count'],
    errors='ignore'
)
print(f"Final dataset size: {len(final_dataset_clean)}")
final_dataset_clean.head(199)